In [ ]:
!pip install boto3 ijson lightgbm catboost xgboost mediapipe==0.10.14 opencv-python --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 3.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grain 0.2.16 requires protobuf>=5.28.3, but you have protobuf 4.25.9 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.9 which is incompatible.
opentelemetry-proto 1.38.0 requires protobuf<7.0,>=5.0, but you have protobuf 4.25

In [ ]:
import json
import cv2
import boto3
import ijson
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import tensorflow as tf
import mediapipe as mp
import gradio as gr
from torch.utils.data import Dataset, DataLoader
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from tqdm import tqdm
from google.colab import userdata
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, f1_score
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [ ]:
def parsing_json_s3():
  ACCESS_KEY = userdata.get("ACCESS_KEY")
  SECRET_KEY = userdata.get("SECRET_KEY")

  s3 = boto3.client(
    "s3",
    endpoint_url="https://storage.yandexcloud.net",
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY
  )

  response = s3.get_object(Bucket="slovo-mediapipe", Key="slovo_mediapipe.json")
  parser = ijson.kvitems(response["Body"], "")

  df = pd.read_csv("/content/annotations.csv", sep="\t")
  attachment_ids = df["attachment_id"].tolist()
  labels = df["text"].tolist()
  id_label_dict = {item: labels[i] for i, item in enumerate(attachment_ids)}

  daktyl = ["А", "Б", "В", "Г", "Д", "Е",
             "Ё", "Ж", "З", "И", "Й", "К",
             "Л", "М", "Н", "О", "П", "Р",
             "С", "Т", "У", "Ф", "Х", "Ц",
             "Ч", "Ш", "Щ", "Ь", "Ы", "Ъ", "Э",
             "Ю", "Я"]

  keys, videos, frames, dots = [], [], [], []
  for key, value in parser:
    if id_label_dict[key] in daktyl:
      keys.append(key)
      for i in range(len(value)):
        item = value[i]["hand 1"]
        for dot in item:
          dots.append(np.float32(dot["x"]))
          dots.append(np.float32(dot["y"]))
          dots.append(np.float32(dot["z"]))
        frames.append(dots)
        dots = []
      videos.append(frames)
      frames = []
  return videos, keys


In [ ]:
def coords_preprocessing(coords: list) -> list:
  scaler = StandardScaler()
  coords_standarded = scaler.fit_transform(coords)
  return coords_standarded

In [ ]:
def features_extraction(coords: list) -> list:
  stat_coords = []
  for i in tqdm(range(len(coords))):
    try:
      video_np = np.array(coords[i], dtype=np.float32)

      video_mean = np.mean(video_np, axis=0)
      video_min = np.min(video_np, axis=0)
      video_max = np.max(video_np, axis=0)
      video_std = np.std(video_np, axis=0)
      video_var = np.var(video_np, axis=0)
      video_range = video_max-video_min
      video_median = np.median(video_np, axis=0)
      video_25 = np.percentile(video_np, 25, axis=0)
      video_75 = np.percentile(video_np, 75, axis=0)


      all_stat = np.array([video_mean,
                        video_min,
                        video_max,
                        video_std,
                        video_var,
                        video_range,
                        video_median,
                        video_25,
                        video_75])
      stat_coords.append(all_stat.flatten())
      all_stat = []
    except:
      print(i, " ", coords[i])
  return stat_coords

In [ ]:
def model_classification(model_name: str, X: list, y: list, target_names: list, labels: list) -> float:
  models = {
        "logistic_regression": LogisticRegression(multi_class="multinomial",
                                                  penalty="l2",
                                                  solver="saga"),
        "svm": SVC(C=1,
                   kernel="linear"),
        "knn": KNeighborsClassifier(n_neighbors=3,
                                    metric="euclidean",
                                    p=2),
        "naive_bayes": GaussianNB(var_smoothing=1e-12),
        "random_forest": RandomForestClassifier(bootstrap=True,
                                                max_depth=30,
                                                max_features="sqrt",
                                                min_samples_leaf=4,
                                                min_samples_split=2,
                                                n_estimators=100),
        "lightgbm": LGBMClassifier(objective="multiclass",
                                   n_estimators=100,
                                   verbose=-1),
        "xgboost": XGBClassifier(objective='multi:softprob',
                                 n_estimators=100,
                                 verbose=-1),
        "catboost": CatBoostClassifier(loss_function='MultiClass',
                                       n_estimators=10,
                                       verbose=0)
    }
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)
  model = models[model_name]
  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)
  accuracy_metric, f1_metric = accuracy_score(y_pred, y_test), f1_score(y_pred, y_test, average="macro", zero_division=0)
  classification_report_metric = classification_report(y_pred, y_test, zero_division=0, target_names=target_names, labels=labels)

  # kfolds = KFold(n_splits=5, random_state=42, shuffle=True)
  # accuracy_metric, f1_metric, classification_report_metric = [], [], []
  # for i, (train_index, test_index) in enumerate(kfolds.split(X, y)):
  #   X_train, X_test = X[train_index], X[test_index]
  #   y_train, y_test = y[train_index], y[test_index]

  #   model = models[model_name]
  #   model.fit(X_train, y_train)
  #   y_pred = model.predict(X_test)
  #   accuracy_metric.append(accuracy_score(y_pred, y_test))
  #   f1_metric.append(f1_score(y_pred, y_test, average="macro"))
  #   classification_report_metric.append(classification_report(y_pred, y_test))

  return accuracy_metric, f1_metric, classification_report_metric

In [ ]:
def x_preprocessing(videos: list):
    video_preprocessed, mistakes = [], []
    for i, video in tqdm(enumerate(videos)):
      try:
        video_np = np.array(video)
        A, F = video_np.shape
        old_time = np.linspace(0, 1, A)
        new_time = np.linspace(0, 1, 48)
        resampled_video = np.zeros((48, 63), dtype=video_np.dtype)
        for f in range(F):
            resampled_video[:, f] = np.interp(
                new_time,
                old_time,
                video_np[:, f]
            )
        resampled_video = resampled_video.reshape(48, 21, 3)
        wrist = resampled_video[:, 0:1, :]
        resampled_video_normalized = resampled_video - wrist
        resampled_video_normalized = resampled_video_normalized.reshape(48, 21, 3)
        video_preprocessed.append(resampled_video_normalized)
      except:
          mistakes.append(i)
    return video_preprocessed, mistakes

In [ ]:
def y_preprocessing(keys: list) -> list:
  df = pd.read_csv("annotations.csv", sep="\t")
  attachments, labels = df["attachment_id"].tolist(), df["text"].tolist()
  labels_dict = {item: labels[i] for i, item in enumerate(attachments)}

  labels_classification = [labels_dict[id] for id in keys]
  encoder = LabelEncoder()
  labels_encoded = np.int32(encoder.fit_transform(labels_classification))
  return labels_encoded, encoder.classes_, encoder

In [ ]:
def hold_out(X: list, y: list):
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)

  train_ds = GestureDataset(X_train, y_train)
  test_ds = GestureDataset(X_test, y_test)

  train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
  test_loader = DataLoader(test_ds, batch_size=32)
  return train_loader, test_loader

In [ ]:
class GestureRNN(nn.Module):
  def __init__(self,
               input_size=63,
               hidden_size=128,
               num_layers=2,
               num_classes=33):
    super().__init__()
    # self.lstm = nn.GRU(
    #   input_size=input_size,
    #   hidden_size=hidden_size,
    #   num_layers=num_layers,
    #   batch_first=True,
    #   bidirectional=True
    # )
    self.lstm = nn.LSTM(
      input_size=input_size,
      hidden_size=hidden_size,
      num_layers=num_layers,
      batch_first=True,
      bidirectional=False
    )
    self.fc = nn.Linear(hidden_size, num_classes)

  def forward(self, x):
    out, _ = self.lstm(x)
    out = out[:, -1, :]
    out = self.fc(out)
    return out

In [ ]:
class GestureDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
class GestureCNN(nn.Module):
  def __init__(self, num_classes=33):
    super().__init__()

    self.conv = nn.Sequential(
      nn.Conv1d(63, 32, kernel_size=3, padding=1),
      nn.ReLU(),
      nn.MaxPool1d(2),

      nn.Conv1d(32, 64, kernel_size=3, padding=1),
      nn.ReLU(),
      nn.MaxPool1d(2),
    )

    self.fc = nn.Sequential(
      nn.Flatten(),
      nn.Linear(64 * 12, 128),
      nn.ReLU(),
      nn.Linear(128, num_classes)
    )

  def forward(self, x):
    x = x.reshape(x.shape[0], 48, 63)
    x = x.permute(0, 2, 1)
    x = self.conv(x)
    x = self.fc(x)
    return x

In [ ]:
data = parsing_json_s3()
coords, keys = data[0], data[1]
print(len(coords))
print(len(coords[0]))
print(len(coords[0][0]))

660
76
63


In [ ]:
X, mistakes = x_preprocessing(videos=coords)

660it [00:00, 1886.47it/s]


In [ ]:
y, classes, label_encoder = y_preprocessing(keys=keys)

In [ ]:
data_hold_out = hold_out(X=X, y=y)
train_loader, test_loader = data_hold_out[0], data_hold_out[1]

print(train_loader)
print(test_loader)

/tmp/ipykernel_4246/123938772.py:3: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  self.X = torch.tensor(X, dtype=torch.float32)


In [ ]:
# model_rnn = GestureRNN()
model_cnn = GestureCNN()

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_cnn.parameters(), lr=1e-3)

In [ ]:
for epoch in range(100):
  for x_train_loader, y_train_loader in train_loader:
    optimizer.zero_grad()
    outputs = model_cnn(x_train_loader)
    loss = criterion(outputs, y_train_loader)
    loss.backward()
    optimizer.step()

  correct, total = 0, 0
  with torch.no_grad():
    for x_test_loader, y_test_loader in test_loader:
      outputs = model_cnn(x_test_loader)
      _, predicted = torch.max(outputs, 1)
      total += y_test_loader.size(0)
      correct += (predicted == y_test_loader).sum().item()
  print(f"Epoch {epoch}. Loss {loss.item()}")
  print(f"Epoch {epoch}. Accuracy {correct/total}")

Epoch 0. Loss 3.462397575378418
Epoch 0. Accuracy 0.045454545454545456
Epoch 1. Loss 3.3982903957366943
Epoch 1. Accuracy 0.05303030303030303
Epoch 2. Loss 2.8231489658355713
Epoch 2. Accuracy 0.05303030303030303
Epoch 3. Loss 3.0064196586608887
Epoch 3. Accuracy 0.09090909090909091
Epoch 4. Loss 2.639965295791626
Epoch 4. Accuracy 0.06818181818181818
Epoch 5. Loss 3.0491135120391846
Epoch 5. Accuracy 0.07575757575757576
Epoch 6. Loss 2.803997039794922
Epoch 6. Accuracy 0.10606060606060606
Epoch 7. Loss 2.8017165660858154
Epoch 7. Accuracy 0.12121212121212122
Epoch 8. Loss 2.387828826904297
Epoch 8. Accuracy 0.17424242424242425
Epoch 9. Loss 2.339348793029785
Epoch 9. Accuracy 0.15151515151515152
Epoch 10. Loss 2.2531120777130127
Epoch 10. Accuracy 0.16666666666666666
Epoch 11. Loss 2.2263081073760986
Epoch 11. Accuracy 0.25
Epoch 12. Loss 1.6946510076522827
Epoch 12. Accuracy 0.23484848484848486
Epoch 13. Loss 1.7587029933929443
Epoch 13. Accuracy 0.29545454545454547
Epoch 14. Loss 1.

In [ ]:
model_cnn.eval()

correct = 0
total = 0

with torch.no_grad():
    for x_test_loader, y_test_loader in test_loader:
        outputs = model_cnn(x_test_loader)

        _, predicted = torch.max(outputs, 1)

        total += y_test_loader.size(0)
        correct += (predicted == y_test_loader).sum().item()

accuracy = correct / total
print(f"Accuracy: {accuracy}")

Accuracy: 0.6287878787878788


# Инференс

In [ ]:
def mediapipe_markdown(video_path: str) -> list:
    mp_hands = mp.solutions.hands
    mp_draw = mp.solutions.drawing_utils

    hands = mp_hands.Hands(
        static_image_mode=False,
        max_num_hands=2,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    )

    cap = cv2.VideoCapture(video_path)
    if cap.isOpened():

        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = cap.get(cv2.CAP_PROP_FPS)

        out = cv2.VideoWriter(
            "output.mp4",
            cv2.VideoWriter_fourcc(*'mp4v'),
            fps,
            (width, height)
        )
        dots, frames, video = [], [], []
        while cap.isOpened():
            success, frame = cap.read()
            if not success:
                break
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = hands.process(rgb_frame)
            if results.multi_hand_landmarks:
                for hand_landmarks in results.multi_hand_landmarks:
                    mp_draw.draw_landmarks(
                        frame,
                        hand_landmarks,
                        mp_hands.HAND_CONNECTIONS,
                        mp_draw.DrawingSpec(color=(0, 255, 0), thickness=2, circle_radius=3),
                        mp_draw.DrawingSpec(color=(255, 0, 0), thickness=2)
                    )
            out.write(frame)

            if results.multi_hand_landmarks:
                for hand_landmarks in results.multi_hand_landmarks:
                    for index, coord in enumerate(hand_landmarks.landmark):
                        dots.append(np.float32(coord.x))
                        dots.append(np.float32(coord.y))
                        dots.append(np.float32(coord.z))
                        frames.append(dots)
                        dots = []
                    video.append(frames)
                    frames = []
        cap.release()
        out.release()
        cv2.destroyAllWindows()
        return video
    else:
        return "Video was not opened"


In [ ]:
def video_preprocessing(video_coords: list) -> list:
    video_preprocessed = []
    video_np = np.array(video_coords)
    A, B, C = video_np.shape
    old_time = np.linspace(0, 1, A)
    new_time = np.linspace(0, 1, 48)
    resampled_video = np.zeros((48, 21, 3), dtype=video_np.dtype)
    for b in range(B):
        for c in range(C):
            resampled_video[:, b, c] = np.interp(
                new_time,
                old_time,
                video_np[:, b, c]
            )
    wrist = resampled_video[:, 0:1, :]
    resampled_video_normalized = resampled_video-wrist
    resampled_video_normalized = resampled_video_normalized.reshape(48, 63)
    video_preprocessed.append(resampled_video_normalized)
    return video_preprocessed

In [ ]:
def pipeline(video_path) -> str:
    video_coords = mediapipe_markdown(video_path=video_path)
    video_coords_preprocessed = video_preprocessing(video_coords=video_coords)

    model_cnn.eval()
    with torch.no_grad():
        outputs = model_cnn(torch.tensor(video_coords_preprocessed, dtype=torch.float32))
        predicted_class = torch.argmax(outputs, dim=1).item()
        probs = torch.softmax(outputs, dim=1)
        confidence = probs[0][predicted_class].item()
        predicted_class_label = label_encoder.inverse_transform([predicted_class])[0]
    return predicted_class_label

In [ ]:
demo = gr.Interface(
    fn=pipeline,
    inputs=gr.Video(label="Видео", height=300, width=500),
    outputs=gr.Label(label="Перевод на русский язык"),
    title="Классификация видео РЖЯ",
    description="Загрузите видео с показанным жестом на РЖЯ или включите веб-камеру для демонстрации жеста"
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d2dbb3c0a3771488af.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Перцептрон

In [ ]:
def dense_model(input_dim, num_classes):
  inputs = layers.Input(shape=(input_dim,))
  x = layers.Dense(128, activation="relu")(inputs)
  x = layers.BatchNormalization()(x)
  x = layers.Dropout(0.3)(x)

  x = layers.Dense(64, activation="relu")(inputs)
  x = layers.BatchNormalization()(x)
  x = layers.Dropout(0.3)(x)

  outputs = layers.Dense(num_classes, activation="softmax")(x)
  model = models.Model(inputs, outputs)
  return model

In [ ]:
def dense_classification(X: list, y: list):
  folds = KFold(n_splits=5, random_state=42, shuffle=True)
  accuracy_all = []
  for i, (train_index, test_index) in enumerate(folds.split(X, y)):
    X_train, X_test = np.array(X)[train_index], np.array(X)[test_index]
    y_train, y_test = np.array(y)[train_index], np.array(y)[test_index]


    X_train_tf, X_test_tf = tf.convert_to_tensor(X_train, dtype=np.float32), tf.convert_to_tensor(X_test, dtype=np.float32)
    y_train_tf, y_test_tf = tf.convert_to_tensor(y_train, dtype=np.int32), tf.convert_to_tensor(y_test, dtype=np.int32)

    model = dense_model(567, 1000)
    model.compile(optimizer="Adam", metrics=["accuracy"], loss='sparse_categorical_crossentropy')

    model.fit(X_train_tf, y_train_tf, epochs=100, batch_size=32, validation_data=(X_test_tf, y_test_tf))
    loss, accuracy = model.evaluate(X_test_tf, y_test_tf)
    accuracy_all.append(accuracy)
  return accuracy_all